# 13 · Testing two hypotheses: smoking and physical activity
Notebook 09 compared the model with the literature and found two oddities, each with an untested explanation:

| Feature | What the model does | Hypothesis we wrote down |
|---|---|---|
| `PhysActivity` | right sign (protective) but only rank 17 of 18 in importance | its information is absorbed by other variables (`GenHlth`, `DiffWalk`, `BMI`...) |
| `Smoker` | **protective** sign, opposite to the literature (RR 1.44) | confounding with age and health status |

Here we test these relationships using two simple approaches:

1. **Nested logistic regressions:** we use the training set and add control variables step by step to see how the odds ratio (OR) of each feature changes. This also allows us to see the initial association before adding any controls.

2. **Model ablation:** we retrain the reduced LightGBM model after removing different groups of variables and check whether the SHAP importance and the direction of each feature change.

**One important caveat for interpreting the results:** variables such as GenHlth, HighBP or HeartDiseaseorAttack may be part of the process linking smoking or physical inactivity to diabetes (for example, smoking → poor health → diabetes). Controlling for these variables removes part of that relationship, so the adjusted effect does not represent the total effect of the habit. We are testing statistical relationships and assumptions about the model, not trying to estimate causal effects.

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import warnings
import contextlib
import io
import statsmodels.api as sm, lightgbm as lgb
from functools import partialmethod
from tqdm import tqdm; tqdm.__init__ = partialmethod(tqdm.__init__, disable=True)
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from src.preprocess import get_splits

warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid")
X_tr, X_va, X_te, y_tr, y_va, y_te = get_splits()
DROP = ["CholCheck", "HvyAlcoholConsump", "AnyHealthcare"]; keep = [c for c in X_tr.columns if c not in DROP]
print(len(keep), "features in the reduced set")

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


18 features in the reduced set


## 1. Crude association by age group
If smoking were only "confounded with age", the raw association should already look different once we compare people of similar age.

In [2]:
age_g = pd.cut(X_tr.Age, [0, 5, 9, 13], labels=["18-44", "45-64", "65+"])
rows = []
for f in ["Smoker", "PhysActivity"]:
    for g in ["all"] + list(age_g.cat.categories):
        m = np.ones(len(X_tr), bool) if g == "all" else (age_g == g).values
        r1, r0 = y_tr[m & (X_tr[f] == 1).values].mean(), y_tr[m & (X_tr[f] == 0).values].mean()
        rows.append({"feature": f, "age group": g, "rate if 1": r1, "rate if 0": r0, "ratio (1 vs 0)": r1 / r0, "n": int(m.sum())})
pd.DataFrame(rows).round(3)

,feature,age group,rate if 1,rate if 0,ratio (1 vs 0),n
0,Smoker,all,0.171,0.137,1.244,160631
1,Smoker,18-44,0.054,0.040,1.369,34125
2,Smoker,45-64,0.163,0.140,1.164,68364
3,Smoker,65+,0.228,0.208,1.097,58142
4,PhysActivity,all,0.131,0.214,0.613,160631
5,PhysActivity,18-44,0.040,0.065,0.622,34125
6,PhysActivity,45-64,0.130,0.206,0.634,68364
7,PhysActivity,65+,0.192,0.282,0.680,58142


## 2. Nested logistic regressions (training set)
Odds ratio of the feature (value 1 vs 0) with 95 % CI, adding blocks of controls. Last row: all 18 features of the reduced model.

In [3]:
blocks = [("crude", []), ("+ Age, Sex", ["Age", "Sex"]), ("+ BMI", ["BMI"]), ("+ HighBP, HighChol", ["HighBP", "HighChol"]),
          ("+ GenHlth, PhysHlth, MentHlth, DiffWalk", ["GenHlth", "PhysHlth", "MentHlth", "DiffWalk"]), ("+ all other reduced features", [c for c in keep])]
def nested(target):
    ctrl, out = [], []
    for name, add in blocks:
        ctrl = [c for c in dict.fromkeys(ctrl + add) if c != target]
        res = sm.Logit(y_tr.values, sm.add_constant(X_tr[[target] + ctrl].astype(float))).fit(disp=0)
        lo, hi = np.exp(res.conf_int().loc[target]); out.append({"controls": name, "OR": np.exp(res.params[target]), "95% CI": f"{lo:.2f}-{hi:.2f}"})
    return pd.DataFrame(out).round(3)
print("Smoker"); display(nested("Smoker")); print("PhysActivity"); display(nested("PhysActivity"))

Smoker


,controls,OR,95% CI
0,crude,1.294,1.26-1.33
1,"+ Age, Sex",1.156,1.12-1.19
2,+ BMI,1.178,1.14-1.21
3,"+ HighBP, HighChol",1.106,1.07-1.14
4,"+ GenHlth, PhysHlth, MentHlth, DiffWalk",0.970,0.94-1.00
5,+ all other reduced features,0.947,0.92-0.98


PhysActivity


,controls,OR,95% CI
0,crude,0.555,0.54-0.57
1,"+ Age, Sex",0.591,0.57-0.61
2,+ BMI,0.700,0.68-0.72
3,"+ HighBP, HighChol",0.743,0.72-0.77
4,"+ GenHlth, PhysHlth, MentHlth, DiffWalk",0.960,0.93-0.99
5,+ all other reduced features,0.972,0.94-1.00


## 3. Ablation of the model
Retrain the reduced LightGBM (same hyper-parameters and early stopping as before) without blocks of variables. For each version: test ROC-AUC, and for Smoker and PhysActivity the mean |SHAP| (5,000 test people, 100-row background), the rank among the remaining features, and the direction (Spearman between the feature value and its SHAP value).

In [ ]:
HEALTH = ["GenHlth", "PhysHlth", "MentHlth", "DiffWalk"]
CLINICAL = HEALTH + ["BMI", "HighBP", "HighChol", "HeartDiseaseorAttack", "Stroke"]
variants = {"base (18 features)": [], "- self-reported health": HEALTH, "- BMI": ["BMI"], "- health and BMI": HEALTH + ["BMI"],
            "- all clinical/health variables": CLINICAL, "- Age": ["Age"]}
idx = np.random.default_rng(0).choice(len(X_te), 5000, replace=False); bg = X_tr.sample(100, random_state=0)
rows = []
for name, drop in variants.items():
    cols = [c for c in keep if c not in drop]
    m = lgb.LGBMClassifier(n_estimators=3000, learning_rate=0.03, num_leaves=15, min_child_samples=100, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.8, reg_lambda=5.0, random_state=42, verbose=-1)
    m.fit(X_tr[cols], y_tr, eval_set=[(X_va[cols], y_va)], eval_metric="binary_logloss", callbacks=[lgb.early_stopping(100, verbose=False)])
    Xs = X_te.iloc[idx][cols]
    with contextlib.redirect_stderr(io.StringIO()):
        sv = shap.TreeExplainer(m, data=bg[cols], feature_perturbation="interventional", model_output="raw")(Xs).values
    S = pd.DataFrame(sv, columns=cols); imp = S.abs().mean().sort_values(ascending=False); o = {"version": name, "n features": len(cols), "test AUC": roc_auc_score(y_te, m.predict_proba(X_te[cols])[:, 1])}
    for f in ["Smoker", "PhysActivity"]:
        o[f"{f} |SHAP|"] = imp[f]; o[f"{f} rank"] = f"{list(imp.index).index(f) + 1}/{len(cols)}"; o[f"{f} direction (rho)"] = spearmanr(Xs[f], S[f])[0]
    rows.append(o); print(name, "done")
abl = pd.DataFrame(rows).set_index("version"); abl.round(3)

base (18 features) done


- self-reported health done


- BMI done


- health and BMI done


- all clinical/health variables done


- Age done


,n features,test AUC,Smoker |SHAP|,Smoker rank,Smoker direction (rho),PhysActivity |SHAP|,PhysActivity rank,PhysActivity direction (rho)
version,,,,,,,,
base (18 features),18,0.815,0.030,11/18,-0.863,0.006,17/18,-0.324
- self-reported health,14,0.796,0.011,12/14,-0.468,0.063,8/14,-0.768
- BMI,17,0.796,0.052,10/17,-0.864,0.029,13/17,-0.750
- health and BMI,13,0.766,0.029,10/13,-0.850,0.107,7/13,-0.768
- all clinical/health variables,9,0.700,0.018,9/9,0.208,0.143,4/9,-0.768
- Age,17,0.805,0.013,15/17,-0.263,0.011,16/17,-0.761


## 4. Conclusion

**Physical activity: the hypothesis is confirmed.**

- **The initial relationship is strong and goes in the expected direction:** the OR is 0.56 (0.54–0.57). People who are physically active have a diabetes rate of 13.1%, compared to 21.4% for inactive people. This pattern is consistent across all age groups, with ratios between 0.62 and 0.68.

- **However, the effect almost disappears when we add self-reported health variables:** the OR goes from 0.74 to 0.96 (0.93–0.99), and reaches 0.97 (0.94–1.00) when all 18 features are included. BMI (0.59 → 0.70) and blood pressure/cholesterol (0.70 → 0.74) also explain part of the initial relationship.

- **The model also gives physical activity very little importance:** its mean |SHAP| is only 0.006 (rank 17/18). When we remove the self-reported health variables, it increases to 0.063 (rank 8/14). Without BMI it reaches 0.029 (13/17), without both groups it goes up to 0.107 (7/13), and without all clinical and health variables it reaches 0.143 (4/9). The direction remains protective in all cases (rho between −0.75 and −0.77).

- **What this means:** the model does not give physical activity little importance because it is irrelevant, but because much of the information it provides is already captured by health status and body weight. Whether physical activity actually causes better health is a different question that these data cannot answer.

**Smoking: the hypothesis is partly confirmed, but saying that it "contradicts the literature" is too strong.**

- **The initial relationship is positive:** the diabetes rate is 17.1% for smokers and 13.7% for non-smokers (OR 1.29, 1.26–1.33). This pattern appears in every age group, with ratios of 1.37 for ages 18–44, 1.16 for 45–64 and 1.10 for 65+.

- **Age explains part of this relationship:** the OR drops to 1.16 after controlling for Age and Sex. BMI does not make much difference, while blood pressure and cholesterol reduce it slightly further to 1.11. Once the self-reported health variables are added, the OR drops to 0.97 (0.94–1.00), and reaches 0.95 (0.92–0.98) with all 18 features.

- **The model shows a similar change in direction:** the negative relationship (rho = −0.86) becomes weaker when the health variables are removed (−0.47) and actually changes to a positive direction (+0.21) when all clinical and health variables are removed. However, this comes with a drop in AUC to 0.70. Removing Age also weakens the negative relationship (−0.26), while removing BMI makes almost no difference.

- **What this means:** the apparently protective effect of smoking is a conditional relationship. In other words, among people of the same age and with similar self-reported health, smokers receive a slightly lower risk score. However, this is not the same effect reported in the scientific literature (RR 1.44), which is closer to the total effect of smoking. There are some possible explanations, although we have not tested them directly. For example, health status may be part of the pathway between smoking and diabetes, so controlling for it removes part of that relationship. Also, smokers who report being in good health may differ from other people in ways that are not captured by the model. Therefore, the model is answering a different question from those studies. This also shows why the direction of a SHAP value should never be interpreted as medical advice.

**Caveats:** these regressions and ablation experiments are descriptive, and removing variables changes how much information the remaining features can capture. After adjustment, the smoking effect is also quite small (mean |SHAP| = 0.03). Finally, all the variables are self-reported.